# XGBoost: local sparse-cell validation
Four configurations: identity/expression inputs, with/without output PCA.
Run each model cell separately. Expression preprocessing matches Pyboost.ipynb.
Requires xgboost >= 2.0, scikit-learn >= 1.2, pandas, numpy and pyarrow.
If needed, install in this notebook kernel with `%pip install "xgboost>=2.0" "scikit-learn>=1.2" pandas numpy pyarrow`.

In [7]:
from pathlib import Path
from google.colab import drive
import sys
import importlib
import pandas as pd

drive.mount("/content/drive")

# Google Drive 中存放数据和 split_data.py 的文件夹
PROJECT_DIR = Path("/content/drive/MyDrive/kaggle_single_cell")
DATA_DIR = PROJECT_DIR

print("split_data.py 是否存在：",
      (PROJECT_DIR / "split_data.py").is_file())

sys.path.insert(0, str(PROJECT_DIR))
importlib.invalidate_caches()

from split_data import make_sparse_splits, apply_split

SEED = 42
PCA_DIM = 160
PARAMS = dict(
    n_estimators=120, max_depth=3, learning_rate=0.06,
    subsample=0.9, colsample_bytree=0.9,
    objective="reg:squarederror", tree_method="hist",
    device="cpu", n_jobs=2, random_state=SEED,
    reg_lambda=1.0, min_child_weight=1.0, max_bin=256,
)
# Keep the original XGBoost output strategies. Both expression variants
# use identical settings within each PCA/no-PCA comparison.
PCA_STRATEGY = "one_output_per_tree"
DIRECT_STRATEGY = "multi_output_tree"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
split_data.py 是否存在： True


In [9]:
# ===== 1. 导入后续代码需要的包 =====
from pathlib import Path
from time import perf_counter
import sys
import gc
import json
import shutil
import subprocess
import importlib

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from IPython.display import display
from google.colab import drive
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA

import xgboost as xgb
from xgboost import XGBRegressor


# ===== 2. 挂载 Google Drive，设置输入与输出路径 =====
drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/kaggle_single_cell")
DATA_DIR = PROJECT_DIR

OUT = PROJECT_DIR / "xgboost_results"
OUT.mkdir(parents=True, exist_ok=True)

# 优先使用已有的 baseline expression 缓存，避免重新计算
PROFILE_PATH = OUT / "baseline_expression_donor_balanced_v2.csv"
PYBOOST_PROFILE_PATH = (
    PROJECT_DIR / "pyboost_results" / PROFILE_PATH.name
)

if not PROFILE_PATH.exists() and PYBOOST_PROFILE_PATH.exists():
    shutil.copy2(PYBOOST_PROFILE_PATH, PROFILE_PATH)
    print("已复制 PyBoost 的 expression 缓存：", PROFILE_PATH)


# ===== 3. 导入数据划分程序 =====
# 支持两种位置：
# ① Google Drive 的 kaggle_single_cell 文件夹
# ② Colab 左侧文件栏上传后的 /content 文件夹
if (PROJECT_DIR / "split_data.py").is_file():
    CODE_DIR = PROJECT_DIR
elif Path("/content/split_data.py").is_file():
    CODE_DIR = Path("/content")
else:
    raise FileNotFoundError(
        "找不到 split_data.py。请将它放在 "
        "/content/drive/MyDrive/kaggle_single_cell/，"
        "或通过 Colab 左侧文件栏上传到 /content。"
    )

sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()

from split_data import make_sparse_splits, apply_split


# ===== 4. 检查 GPU 和 XGBoost =====
print("XGBoost 版本：", xgb.__version__)

if tuple(map(int, xgb.__version__.split(".")[:2])) < (3, 4):
    raise RuntimeError(
        "当前 XGBoost 版本过旧。请先运行之前的 "
        "XGBoost 3.4.2 安装单元格，重启会话，再运行本单元格。"
    )

gpu_info = subprocess.run(
    ["nvidia-smi", "-L"],
    capture_output=True,
    text=True,
)
if gpu_info.returncode != 0:
    raise RuntimeError(
        "未能访问 GPU。请在 Colab 的“运行时 → 更改运行时类型”中选择 GPU。"
    )
print(gpu_info.stdout.strip())

if not xgb.build_info().get("USE_CUDA", False):
    raise RuntimeError(
        "当前 XGBoost 安装不支持 CUDA，请使用前面的 xgboost-cu12 安装单元格。"
    )


# ===== 5. 原有模型参数：仅使用 GPU，其他设置保持一致 =====
SEED = 42
PCA_DIM = 160

PARAMS = dict(
    n_estimators=120,
    max_depth=3,
    learning_rate=0.06,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    tree_method="hist",
    device="cuda",
    n_jobs=2,
    random_state=SEED,
    reg_lambda=1.0,
    min_child_weight=1.0,
    max_bin=256,
)

PCA_STRATEGY = "one_output_per_tree"
DIRECT_STRATEGY = "multi_output_tree"


# ===== 6. 用小数据检查两种策略是否真正使用 GPU =====
# 后面的训练函数也会调用这个检查函数，因此需要保留。
def check_cuda_model(model):
    config = json.loads(model.get_booster().save_config())
    actual_device = config["learner"]["generic_param"]["device"]

    if not actual_device.startswith("cuda"):
        raise RuntimeError(
            f"模型实际使用的是 {actual_device}，没有使用 GPU。"
            "请检查 Colab 的 GPU 设置。"
        )


probe_x = np.random.default_rng(SEED).normal(
    size=(32, 4)
).astype(np.float32)

probe_y = np.column_stack([
    probe_x[:, 0],
    probe_x[:, 1],
]).astype(np.float32)

for strategy in (PCA_STRATEGY, DIRECT_STRATEGY):
    probe = XGBRegressor(
        **{**PARAMS, "n_estimators": 2},
        multi_strategy=strategy,
    )
    probe.fit(probe_x, probe_y)
    check_cuda_model(probe)

    probe_prediction = probe.get_booster().predict(
        xgb.DMatrix(probe_x)
    )
    assert probe_prediction.shape == probe_y.shape
    assert np.isfinite(probe_prediction).all()

    print("GPU 检查通过：", strategy)
    del probe, probe_prediction

del probe_x, probe_y
gc.collect()

print("\n初始化完成。")
print("数据目录：", DATA_DIR)
print("划分程序：", CODE_DIR / "split_data.py")
print("结果目录：", OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
已复制 PyBoost 的 expression 缓存： /content/drive/MyDrive/kaggle_single_cell/xgboost_results/baseline_expression_donor_balanced_v2.csv
XGBoost 版本： 3.4.1
GPU 0: Tesla T4 (UUID: GPU-2b1e1106-23a4-7428-55c2-c5428daa9252)
GPU 检查通过： one_output_per_tree
GPU 检查通过： multi_output_tree

初始化完成。
数据目录： /content/drive/MyDrive/kaggle_single_cell
划分程序： /content/drive/MyDrive/kaggle_single_cell/split_data.py
结果目录： /content/drive/MyDrive/kaggle_single_cell/xgboost_results


## 1. Read local data

In [10]:
df = pd.read_parquet(DATA_DIR / "de_train.parquet").reset_index(drop=True)
idmap = pd.read_csv(DATA_DIR / "id_map.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv", usecols=["id"])
genes = pd.read_csv(DATA_DIR / "sample_submission.csv", nrows=0).columns.drop("id").tolist()
pairs = ["cell_type", "sm_name"]
assert len(df) == 614 and len(genes) == 18211
assert not df.duplicated(pairs).any()
assert df[pairs].notna().all().all()
assert idmap["id"].is_unique and sample["id"].is_unique
assert set(idmap["id"]) == set(sample["id"])
idmap = idmap.set_index("id").loc[sample["id"]].reset_index()
Y = df[genes].to_numpy(dtype=np.float32)
assert np.isfinite(Y).all()
assert not set(map(tuple, df[pairs].values)) & set(map(tuple, idmap[pairs].values))
assert set(idmap.sm_name).issubset(set(df.sm_name))
print("Training targets:", Y.shape, "| Official test pairs:", len(idmap))
display(df.groupby("cell_type").size().rename("pairs").to_frame())

Training targets: (614, 18211) | Official test pairs: 255


,pairs
cell_type,
B cells,17
Myeloid cells,17
NK cells,146
T cells CD4+,146
T cells CD8+,142
T regulatory cells,146


## 2. Four sparse-cell folds
Use the same split function as Model_workflow.ipynb. No split files are rewritten.

In [11]:
folds = []
for cell, manifest in make_sparse_splits(df).items():
    train, validation = apply_split(df, manifest)
    folds.append((cell, train, validation))
display(pd.DataFrame([
    {"cell_type": cell, "train": len(tr), "validation": len(va)}
    for cell, tr, va in folds
]))

,cell_type,train,validation
0,NK cells,485,129
1,T cells CD4+,485,129
2,T cells CD8+,487,127
3,T regulatory cells,485,129


## 3. Baseline expression (same preprocessing as PyBoost)
Only RNA Gene Expression features are used. Average within donor × cell type,
including sparse zeros, then average donors equally. Input expression is not PCA-reduced.
The first expression experiment builds the cache if absent; identity-only runs skip this step.

In [12]:
profiles = None

def load_expression_profiles():
    MULTIOME_DIR = DATA_DIR
    OUT.mkdir(parents=True, exist_ok=True)
    PROFILE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if PROFILE_PATH.exists():
        profiles = pd.read_csv(
            PROFILE_PATH,
            index_col=0
        )

        print("\nLoaded cached baseline profiles:", profiles.shape)

    else:
        required = [
            "multiome_obs_meta.csv",
            "multiome_var_meta.csv",
            "multiome_train.parquet",
        ]

        missing = [
            name for name in required
            if not (MULTIOME_DIR / name).is_file()
        ]

        if missing:
            raise FileNotFoundError(
                "本地数据目录中缺少：" + ", ".join(missing)
                + "\n当前目录：" + str(MULTIOME_DIR)
            )

        meta = pd.read_csv(
            MULTIOME_DIR / "multiome_obs_meta.csv"
        )
        var = pd.read_csv(
            MULTIOME_DIR / "multiome_var_meta.csv"
        )

        # Handle CSV index columns.
        if "obs_id" not in meta.columns:
            first = meta.columns[0]
            if first.startswith("Unnamed:"):
                meta.rename(
                    columns={first: "obs_id"},
                    inplace=True
                )

        if "location" not in var.columns:
            first = var.columns[0]
            if first.startswith("Unnamed:"):
                var.rename(
                    columns={first: "location"},
                    inplace=True
                )

        assert {
            "obs_id", "donor_id", "cell_type"
        }.issubset(meta.columns), meta.columns.tolist()

        assert {
            "location", "feature_type"
        }.issubset(var.columns), var.columns.tolist()

        assert meta[
            ["obs_id", "donor_id", "cell_type"]
        ].notna().all().all()

        meta["obs_id"] = meta["obs_id"].astype(str)

        assert not meta["obs_id"].duplicated().any()

        expr_genes = (
            var.loc[
                var["feature_type"].eq("Gene Expression"),
                "location"
            ]
            .astype(str)
            .drop_duplicates()
            .tolist()
        )

        assert expr_genes, \
            "没有找到 Gene Expression 特征，请检查 var 表。"

        gene_index = pd.Series(
            np.arange(len(expr_genes)),
            index=expr_genes
        )

        meta["group"] = (
            meta.groupby(
                ["donor_id", "cell_type"],
                sort=True
            ).ngroup()
        )

        group_meta = (
            meta[["group", "donor_id", "cell_type"]]
            .drop_duplicates()
            .sort_values("group")
            .reset_index(drop=True)
        )

        n_cells = (
            meta.groupby("group")
            .size()
            .reindex(group_meta["group"])
            .to_numpy()
        )

        obs_group = meta.set_index("obs_id")["group"]

        sums = np.zeros(
            (len(group_meta), len(expr_genes)),
            dtype=np.float64
        )

        parquet = pq.ParquetFile(
            MULTIOME_DIR / "multiome_train.parquet"
        )

        expected = {
            "obs_id", "location", "normalized_count"
        }

        assert expected.issubset(
            parquet.schema_arrow.names
        ), parquet.schema_arrow.names

        print("\nBuilding baseline expression profiles...")

        for batch_no, batch in enumerate(
            parquet.iter_batches(
                batch_size=500_000,
                columns=[
                    "obs_id",
                    "location",
                    "normalized_count",
                ]
            )
        ):
            frame = batch.to_pandas()

            gene_ids = (
                frame["location"]
                .astype(str)
                .map(gene_index)
            )

            keep = gene_ids.notna()

            if keep.any():
                selected = frame.loc[keep]

                group_ids = (
                    selected["obs_id"]
                    .astype(str)
                    .map(obs_group)
                )

                assert group_ids.notna().all(), \
                    "存在无法匹配 metadata 的 RNA obs_id。"

                values = selected[
                    "normalized_count"
                ].to_numpy(dtype=np.float64)

                assert np.isfinite(values).all()
                assert (values >= 0).all()

                np.add.at(
                    sums,
                    (
                        group_ids.to_numpy(dtype=int),
                        gene_ids.loc[keep].to_numpy(dtype=int),
                    ),
                    values
                )

            if batch_no % 20 == 0:
                print("Processed batch:", batch_no)

        donor_means = sums / n_cells[:, None]

        profiles = pd.DataFrame(
            donor_means,
            columns=expr_genes
        )
        profiles["cell_type"] = (
            group_meta["cell_type"].to_numpy()
        )

        profiles = (
            profiles.groupby("cell_type", sort=True)
            .mean()
        )

        assert np.isfinite(profiles.to_numpy()).all()

        profiles.to_csv(PROFILE_PATH)

        group_meta.assign(n_cells=n_cells).to_csv(
            OUT / "baseline_group_counts_v2.csv",
            index=False
        )

        np.savez_compressed(
            OUT / "baseline_donor_profiles_v2.npz",
            means=donor_means,
            genes=np.asarray(expr_genes, dtype=str)
        )

        del sums, donor_means, frame
        gc.collect()

        print("Saved:", PROFILE_PATH)

    assert profiles.index.is_unique
    assert profiles.columns.is_unique
    assert set(df["cell_type"]).issubset(profiles.index)
    assert set(idmap["cell_type"]).issubset(profiles.index)
    assert np.isfinite(profiles.to_numpy()).all()

    print("\nBaseline expression features:", profiles.shape)

    return profiles

## 4. Shared training and evaluation
Squared-error training; MRRMSE after reconstruction in gene space.
Each fold fits a fresh encoder, PCA and XGBoost model.
Dense float32 inputs match PyBoost: zero is a real feature value, not missing.
The no-PCA model now uses max_bin=256 rather than 4 to retain continuous expression detail.
XGBoost retains different tree strategies for PCA and no PCA; their comparison is not a pure PCA ablation.

In [13]:
def mrrmse(y_true, y_pred):
    difference = np.asarray(y_true, dtype=np.float64) - np.asarray(y_pred, dtype=np.float64)
    return float(np.sqrt(np.mean(difference ** 2, axis=1)).mean())

def make_features(train, query, use_expression):
    global profiles
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
    a = encoder.fit_transform(train[pairs].astype(str))
    b = encoder.transform(query[pairs].astype(str))
    if use_expression:
        if profiles is None:
            profiles = load_expression_profiles()
        a = np.concatenate([a, profiles.loc[train.cell_type.to_numpy()].to_numpy(dtype=np.float32)], axis=1)
        b = np.concatenate([b, profiles.loc[query.cell_type.to_numpy()].to_numpy(dtype=np.float32)], axis=1)
    return np.ascontiguousarray(a), np.ascontiguousarray(b)

def fit_predict(train, query, use_expression, use_pca, settings):
    a, b = make_features(train, query, use_expression)
    targets = train[genes].to_numpy(dtype=np.float32)
    reducer = None
    if use_pca:
        reducer = PCA(n_components=min(settings["pca_dim"], len(train)-1, len(genes)),
                      svd_solver="randomized", random_state=SEED)
        targets = reducer.fit_transform(targets).astype(np.float32)
    model = XGBRegressor(**settings["params"], multi_strategy=settings["strategy"])
    print("Input:", a.shape, "| Training targets:", targets.shape)
    model.fit(a, targets)
    prediction = model.predict(b)
    if reducer is not None:
        prediction = reducer.inverse_transform(prediction)
    assert prediction.shape == (len(query), len(genes))
    assert np.isfinite(prediction).all()
    return prediction

mean_scores = {}
fold_scores = {}
validated_settings = {}

def evaluate_method(name, use_expression, use_pca, pca_dim=PCA_DIM, **overrides):
    settings = {
        "params": {**PARAMS, **overrides}, "pca_dim": pca_dim,
        "strategy": PCA_STRATEGY if use_pca else DIRECT_STRATEGY,
        "use_expression": use_expression, "use_pca": use_pca,
    }
    # Remove stale results if a rerun fails part-way through.
    for registry in (mean_scores, fold_scores, validated_settings):
        registry.pop(name, None)
    rows = []
    for cell, train, validation in folds:
        print("Validation:", cell, flush=True)
        started = perf_counter()
        prediction = fit_predict(train, validation, use_expression, use_pca, settings)
        rows.append({"cell_type": cell, "n_validation": len(validation),
                     "MRRMSE": mrrmse(validation[genes], prediction),
                     "seconds": perf_counter()-started})
        del prediction
        gc.collect()
    fold_scores[name] = pd.DataFrame(rows)
    mean_scores[name] = float(fold_scores[name]["MRRMSE"].mean())
    validated_settings[name] = settings
    display(fold_scores[name])
    print(f"{name}: four-fold mean MRRMSE = {mean_scores[name]:.6f}")

## Identity + PCA

In [14]:
evaluate_method("identity_pca", use_expression=False, use_pca=True)

Validation: NK cells
Input: (485, 152) | Training targets: (485, 160)
Validation: T cells CD4+
Input: (485, 152) | Training targets: (485, 160)
Validation: T cells CD8+
Input: (487, 152) | Training targets: (487, 160)
Validation: T regulatory cells
Input: (485, 152) | Training targets: (485, 160)


,cell_type,n_validation,MRRMSE,seconds
0,NK cells,129,1.166216,7.900713
1,T cells CD4+,129,1.007232,11.802130
2,T cells CD8+,127,0.846441,7.831494
3,T regulatory cells,129,1.086601,10.127151


identity_pca: four-fold mean MRRMSE = 1.026623


## Expression + PCA

In [14]:
evaluate_method("expression_pca", use_expression=True, use_pca=True)

Validation: NK cells

Loaded cached baseline profiles: (6, 22787)

Baseline expression features: (6, 22787)
Input: (485, 22939) | Training targets: (485, 160)


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [17:19:35] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Validation: T cells CD4+
Input: (485, 22939) | Training targets: (485, 160)
Validation: T cells CD8+
Input: (487, 22939) | Training targets: (487, 160)
Validation: T regulatory cells
Input: (485, 22939) | Training targets: (485, 160)


,cell_type,n_validation,MRRMSE,seconds
0,NK cells,129,1.164561,123.390111
1,T cells CD4+,129,1.003004,122.289356
2,T cells CD8+,127,0.838492,123.872320
3,T regulatory cells,129,1.123966,121.547617


expression_pca: four-fold mean MRRMSE = 1.032506


## Identity without PCA

In [17]:
evaluate_method("identity_no_pca", use_expression=False, use_pca=False)

Validation: NK cells
Input: (485, 152) | Training targets: (485, 18211)
Validation: T cells CD4+
Input: (485, 152) | Training targets: (485, 18211)
Validation: T cells CD8+
Input: (487, 152) | Training targets: (487, 18211)
Validation: T regulatory cells
Input: (485, 152) | Training targets: (485, 18211)


,cell_type,n_validation,MRRMSE,seconds
0,NK cells,129,1.206382,192.473197
1,T cells CD4+,129,1.021581,191.467840
2,T cells CD8+,127,0.832599,192.011156
3,T regulatory cells,129,1.040138,191.914023


identity_no_pca: four-fold mean MRRMSE = 1.025175


## Expression without PCA

In [19]:
evaluate_method("expression_no_pca", use_expression=True, use_pca=False)

Validation: NK cells
Input: (485, 22939) | Training targets: (485, 18211)


XGBoostError: [17:47:33] /__w/xgboost/xgboost/src/tree/updater_gpu_hist.cuh:248: Check failed: n_total_bins < std::numeric_limits<bst_bin_t>::max() (2423046394 vs. 2147483647) : Too many histogram bins: n_total_bins = total_bins * n_targets
Stack trace:
  [bt] (0) /usr/local/lib/python3.13/dist-packages/xgboost/lib/libxgboost.so(+0x21ec5c) [0x7e058ba1ec5c]
  [bt] (1) /usr/local/lib/python3.13/dist-packages/xgboost/lib/libxgboost.so(+0x1166811) [0x7e058c966811]
  [bt] (2) /usr/local/lib/python3.13/dist-packages/xgboost/lib/libxgboost.so(+0x1198f51) [0x7e058c998f51]
  [bt] (3) /usr/local/lib/python3.13/dist-packages/xgboost/lib/libxgboost.so(+0x119e4ca) [0x7e058c99e4ca]
  [bt] (4) /usr/local/lib/python3.13/dist-packages/xgboost/lib/libxgboost.so(+0x640621) [0x7e058be40621]
  [bt] (5) /usr/local/lib/python3.13/dist-packages/xgboost/lib/libxgboost.so(+0x641ff8) [0x7e058be41ff8]
  [bt] (6) /usr/local/lib/python3.13/dist-packages/xgboost/lib/libxgboost.so(+0x688af7) [0x7e058be88af7]
  [bt] (

## Four-fold summary

In [ ]:
summary = pd.Series(mean_scores, name="four_fold_mean_MRRMSE").rename_axis("model").sort_values().to_frame()
display(summary)
if fold_scores:
    OUT.mkdir(parents=True, exist_ok=True)
    summary.to_csv(OUT / "cv_summary.csv")
    pd.concat(fold_scores, names=["model", "fold"]).to_csv(OUT / "cv_scores.csv")

## Official test predictions
Run this cell when ready. Refit each successfully evaluated configuration on all 614 labeled pairs,
using its recorded validation settings. Save CSVs in submissions/xgboost.
To export only selected models, replace list(validated_settings) with their names.

In [20]:
import json
SUBMISSION_DIR = PROJECT_DIR / "submissions" / "xgboost"
models_to_submit = list(validated_settings)
assert models_to_submit, "Run at least one model evaluation cell first."
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
for name in models_to_submit:
    settings = validated_settings[name]
    prediction = fit_predict(df, idmap, settings["use_expression"], settings["use_pca"], settings)
    submission = pd.DataFrame(prediction, columns=genes)
    submission.insert(0, "id", idmap["id"].to_numpy())
    assert submission.shape == (len(idmap), len(genes)+1)
    assert submission["id"].equals(sample["id"])
    destination = SUBMISSION_DIR / f"xgboost_{name}_submission.csv"
    submission.to_csv(destination, index=False, float_format="%.8g")
    destination.with_suffix(".json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
    print("Saved:", destination)
    del prediction, submission
    gc.collect()

Input: (614, 22939) | Training targets: (614, 160)
Saved: /content/drive/MyDrive/kaggle_single_cell/submissions/xgboost/xgboost_expression_pca_submission.csv
Input: (614, 152) | Training targets: (614, 18211)
Saved: /content/drive/MyDrive/kaggle_single_cell/submissions/xgboost/xgboost_identity_no_pca_submission.csv
